# <center> Sample code for single-image reconstruction of exotic synthetic objects </center>

This notebook follows the same structure as the stack reconstruction example, but focuses on challenging **single-hologram** reconstructions.

The workflow is:

1) Build several exotic synthetic objects that are intentionally hard to reconstruct (strong phase gradients, irregular masks, porous textures).
2) Simulate a **single hologram image** per object.
3) Reconstruct each hologram from that single image using `HoloParams.from_hologram` + `HoloReconstructUDF`.
4) Compare reconstructed amplitude/phase against the synthetic ground truth using visual inspection and quantitative metrics (`SSIM`, `PSNR`).

In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter
from skimage.draw import polygon
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.restoration import unwrap_phase

from libertem import api
from libertem.executor.inline import InlineJobExecutor
from libertem.io.dataset.memory import MemoryDataSet

from libertem_holo.base.generate import hologram_frame
from libertem_holo.base.utils import HoloParams
from libertem_holo.udf import HoloReconstructUDF

### Required Functions

In [ ]:
ctx = api.Context(executor=InlineJobExecutor())


def normalize_img(img):
    lo, hi = np.percentile(img, [1, 99])
    if hi <= lo:
        return np.zeros_like(img)
    return np.clip((img - lo) / (hi - lo), 0, 1)


def fit_linear(target, source):
    a, b = np.linalg.lstsq(
        np.stack([source.ravel(), np.ones(source.size)], axis=1),
        target.ravel(),
        rcond=None,
    )[0]
    return a * source + b


def best_phase_match(phase_true, phase_rec):
    # Account for unknown global sign/scale/offset in single-image reconstruction
    cand_pos = fit_linear(phase_true, phase_rec)
    cand_neg = fit_linear(phase_true, -phase_rec)
    ssim_pos = ssim(normalize_img(phase_true), normalize_img(cand_pos), data_range=1.0)
    ssim_neg = ssim(normalize_img(phase_true), normalize_img(cand_neg), data_range=1.0)
    return cand_pos if ssim_pos >= ssim_neg else cand_neg


def quality_metrics(true_img, rec_img):
    true_n = normalize_img(true_img)
    rec_n = normalize_img(rec_img)
    return {
        'ssim': ssim(true_n, rec_n, data_range=1.0),
        'psnr': psnr(true_n, rec_n, data_range=1.0),
    }


def make_star_mask(shape, center, r_outer, r_inner, points=7):
    cy, cx = center
    angles = np.linspace(0, 2 * np.pi, num=points * 2, endpoint=False)
    radii = np.where(np.arange(points * 2) % 2 == 0, r_outer, r_inner)
    ys = cy + radii * np.sin(angles)
    xs = cx + radii * np.cos(angles)
    rr, cc = polygon(ys, xs, shape=shape)
    mask = np.zeros(shape, dtype=bool)
    mask[rr, cc] = True
    return mask


def exotic_objects(shape=(256, 256), seed=4):
    rng = np.random.default_rng(seed)
    sy, sx = shape
    yy, xx = np.meshgrid(np.arange(sy), np.arange(sx), indexing='ij')
    cy, cx = sy / 2, sx / 2
    rr = np.hypot(xx - cx, yy - cy)
    th = np.arctan2(yy - cy, xx - cx)

    objects = []

    # Object 1: distorted vortex ring
    amp1 = 1.0 - 0.45 * np.exp(-((rr - 55) ** 2) / (2 * 11 ** 2)) * (1 + 0.35 * np.cos(6 * th))
    amp1 = np.clip(amp1, 0.15, 1.3)
    phase1 = 2.1 * np.tanh((rr - 48) / 9) + 1.4 * np.sin(3 * th) * np.exp(-(rr / 95) ** 2)
    objects.append(("distorted_vortex_ring", amp1, phase1))

    # Object 2: star polygon with rough interior phase
    mask2 = make_star_mask(shape, center=(0.42 * sy, 0.55 * sx), r_outer=70, r_inner=30, points=8)
    texture2 = gaussian_filter(rng.normal(size=shape), sigma=2.2)
    amp2 = np.ones(shape)
    amp2[mask2] = 0.35 + 0.15 * normalize_img(texture2[mask2])
    phase2 = 0.7 * np.sin(0.14 * xx) + 0.6 * np.cos(0.19 * yy)
    phase2 += 2.6 * gaussian_filter(mask2.astype(float), sigma=2.5)
    objects.append(("star_polygon_with_texture", amp2, phase2))

    # Object 3: porous labyrinth with mixed-frequency phase
    noise3 = gaussian_filter(rng.normal(size=shape), sigma=3.0)
    pores3 = noise3 > np.percentile(noise3, 60)
    amp3 = np.ones(shape)
    amp3[pores3] = 0.4
    amp3 = gaussian_filter(amp3, sigma=0.8)
    phase3 = 1.8 * gaussian_filter(pores3.astype(float), sigma=2.0)
    phase3 += 0.9 * np.sin(0.09 * xx + 0.07 * yy) + 0.8 * np.sin(0.23 * yy)
    objects.append(("porous_labyrinth", amp3, phase3))

    return objects

## Data set

The examples below are synthetic and generated directly in this notebook, so no external files are required.

In [ ]:
objects = exotic_objects(shape=(256, 256), seed=7)
len(objects), [name for name, *_ in objects]

## Step 1: Selecting the holography parameters

For each synthetic object, generate one hologram and estimate sideband parameters directly from that hologram.

In [ ]:
simulations = []

for name, amp_true, phase_true in objects:
    holo = hologram_frame(
        amp=amp_true,
        phi=phase_true,
        counts=4_000,
        sampling=5.0,
        visibility=0.75,
        f_angle=33.0,
        gaussian_noise=0.8,
        poisson_noise=0.20,
    )
    params = HoloParams.from_hologram(
        hologram=holo,
        out_shape=(256, 256),
        line_filter_width=3,
        sb='upper',
    )
    simulations.append({
        'name': name,
        'amp_true': amp_true,
        'phase_true': phase_true,
        'holo': holo,
        'params': params,
    })

len(simulations)

In [ ]:
fig, axes = plt.subplots(len(simulations), 3, figsize=(12, 3.6 * len(simulations)))
if len(simulations) == 1:
    axes = np.array([axes])

for row, sim in enumerate(simulations):
    amp_true = sim['amp_true']
    phase_true = sim['phase_true']
    holo = sim['holo']

    axes[row, 0].imshow(amp_true, cmap='gray')
    axes[row, 0].set_title(f"{sim['name']} | input amplitude")
    axes[row, 0].set_axis_off()

    axes[row, 1].imshow(phase_true, cmap='viridis')
    axes[row, 1].set_title("input phase")
    axes[row, 1].set_axis_off()

    axes[row, 2].imshow(holo, cmap='gray')
    axes[row, 2].set_title("single hologram")
    axes[row, 2].set_axis_off()

fig.tight_layout()

## Step 2: Reconstruction of single holograms to complex wave images

In [ ]:
results = []

for sim in simulations:
    params = sim['params']
    ds_holo = MemoryDataSet(data=sim['holo'][None, ...], sig_dims=2)
    udf = HoloReconstructUDF(
        out_shape=params.out_shape,
        sb_position=params.sb_position,
        aperture=params.aperture,
    )
    wave = ctx.run_udf(dataset=ds_holo, udf=udf)['wave'].data[0]

    amp_rec = np.abs(wave)
    phase_rec = unwrap_phase(np.angle(wave))
    phase_fit = best_phase_match(sim['phase_true'], phase_rec)

    amp_metrics = quality_metrics(sim['amp_true'], fit_linear(sim['amp_true'], amp_rec))
    phase_metrics = quality_metrics(sim['phase_true'], phase_fit)

    results.append({
        **sim,
        'wave': wave,
        'amp_rec': amp_rec,
        'phase_rec': phase_rec,
        'phase_fit': phase_fit,
        'amp_metrics': amp_metrics,
        'phase_metrics': phase_metrics,
    })

len(results)

In [ ]:
fig, axes = plt.subplots(len(results), 4, figsize=(14, 3.6 * len(results)))
if len(results) == 1:
    axes = np.array([axes])

for row, res in enumerate(results):
    axes[row, 0].imshow(res['amp_true'], cmap='gray')
    axes[row, 0].set_title(f"{res['name']} | amp (true)")
    axes[row, 0].set_axis_off()

    axes[row, 1].imshow(res['amp_rec'], cmap='gray')
    axes[row, 1].set_title("amp (reconstructed)")
    axes[row, 1].set_axis_off()

    axes[row, 2].imshow(res['phase_true'], cmap='viridis')
    axes[row, 2].set_title("phase (true)")
    axes[row, 2].set_axis_off()

    axes[row, 3].imshow(res['phase_fit'], cmap='viridis')
    axes[row, 3].set_title("phase (reconstructed, fitted)")
    axes[row, 3].set_axis_off()

fig.tight_layout()

## Step 3: Quantitative summary and takeaways

In [ ]:
for res in results:
    a = res['amp_metrics']
    p = res['phase_metrics']
    print(
        f"{res['name']}: "
        f"amp SSIM={a['ssim']:.3f}, amp PSNR={a['psnr']:.2f} dB | "
        f"phase SSIM={p['ssim']:.3f}, phase PSNR={p['psnr']:.2f} dB"
    )

Observations from this single-image test set:

- All three objects are recoverable from one hologram, but quality varies significantly with object complexity.
- The **porous labyrinth** tends to be the hardest due to broad spatial-frequency content and fine-scale discontinuities.
- For single-image reconstructions, global phase sign/scale/offset ambiguity is expected; fitting helps compare structure against ground truth.
- Next steps can include parameter sweeps (`sampling`, `visibility`, noise level, filter widths) and optional reference-based correction when available.